# 🚗 Uber Rides Advanced Analysis
**1,156 Business Trip Records — 2016**

---

In [ ]:
import pandas as pd, numpy as np, calendar
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings; warnings.filterwarnings('ignore')
print("Libraries loaded")

## 1. Load & Transform Dataset

In [ ]:
df = pd.read_csv("uber_trips.csv")
df.columns = [c.strip() for c in df.columns]
df["START_DATE*"] = pd.to_datetime(df["START_DATE*"], format="%m/%d/%Y %H:%M", errors="coerce")
df["END_DATE*"]   = pd.to_datetime(df["END_DATE*"],   format="%m/%d/%Y %H:%M", errors="coerce")
df = df.dropna(subset=["START_DATE*","END_DATE*"])

df["HOUR"]     = df["START_DATE*"].dt.hour
df["DAY"]      = df["START_DATE*"].dt.day
df["WEEKDAY"]  = df["START_DATE*"].dt.day_name()
df["MONTH"]    = df["START_DATE*"].dt.month
df["MONTH_NAME"] = df["START_DATE*"].dt.month_name()
df["MILES*"]   = pd.to_numeric(df["MILES*"], errors="coerce")
travel_hrs     = (df["END_DATE*"] - df["START_DATE*"]).dt.seconds / 3600
df["TRAVEL_HRS"] = travel_hrs.replace(0, np.nan)
df["SPEED_MPH"]  = df["MILES*"] / df["TRAVEL_HRS"]

print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()

## 2. Summary Statistics

In [ ]:
print("=== UBER RIDES SUMMARY ===")
print(f"Total trips        : {len(df):,}")
print(f"Total miles        : {df['MILES*'].sum():,.1f}")
print(f"Avg miles/trip     : {df['MILES*'].mean():.2f}")
print(f"Avg travel time    : {df['TRAVEL_HRS'].mean()*60:.1f} min")
print(f"Avg speed          : {df['SPEED_MPH'].dropna().mean():.1f} mph")
print(f"\nCategory breakdown:")
print(df["CATEGORY*"].value_counts())
print(f"\nTop 5 purposes:")
print(df["PURPOSE*"].value_counts().head())

## 3. Trip Category & Purpose

In [ ]:
fig = make_subplots(rows=1, cols=2, specs=[[{"type":"pie"},{"type":"bar"}]])
cat_c = df["CATEGORY*"].value_counts()
fig.add_trace(go.Pie(labels=cat_c.index, values=cat_c.values, hole=0.4, name="Category"), row=1, col=1)
fig.update_layout(title="Trip Category Distribution", height=380)
fig.show()

purp_c = df["PURPOSE*"].value_counts().reset_index()
purp_c.columns = ["Purpose","Count"]
fig2 = px.bar(purp_c, x="Count", y="Purpose", orientation="h",
              color="Count", color_continuous_scale="Blues",
              title="Trips by Purpose")
fig2.update_layout(height=450, yaxis=dict(autorange="reversed"))
fig2.show()

## 4. Miles Distribution

In [ ]:
fig = px.histogram(df, x="MILES*", nbins=40,
                   color="CATEGORY*", opacity=0.7, barmode="overlay",
                   title="Trip Distance Distribution (Miles)")
fig.update_layout(height=420)
fig.show()

print(f"Short trips (<5 miles): {(df['MILES*']<5).mean()*100:.1f}%")
print(f"Long trips (>20 miles): {(df['MILES*']>20).mean()*100:.1f}%")

## 5. Time Patterns

In [ ]:
WEEKDAY_ORDER = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
MONTH_ORDER   = ["January","February","March","April","May","June",
                 "July","August","September","October","November","December"]

fig = make_subplots(rows=1, cols=2,
    subplot_titles=["Trips by Hour of Day","Trips by Day of Week"])

hour_c = df.groupby("HOUR").size().reset_index(name="Trips")
wd_c   = df.groupby("WEEKDAY").size().reset_index(name="Trips")
wd_c["WEEKDAY"] = pd.Categorical(wd_c["WEEKDAY"], WEEKDAY_ORDER, ordered=True)
wd_c = wd_c.sort_values("WEEKDAY")

fig.add_trace(go.Bar(x=hour_c["HOUR"], y=hour_c["Trips"],
    marker_color="steelblue", name="By Hour"), row=1, col=1)
fig.add_trace(go.Bar(x=wd_c["WEEKDAY"], y=wd_c["Trips"],
    marker_color="seagreen", name="By Weekday"), row=1, col=2)
fig.update_layout(height=420, showlegend=False)
fig.show()

## 6. Day × Hour Heatmap

In [ ]:
heat_df = df.groupby(["WEEKDAY","HOUR"]).size().reset_index(name="Trips")
heat_pivot = heat_df.pivot(index="WEEKDAY", columns="HOUR", values="Trips").fillna(0)
heat_pivot = heat_pivot.reindex(
    [d for d in WEEKDAY_ORDER if d in heat_pivot.index])
fig = px.imshow(heat_pivot, color_continuous_scale="Blues",
                text_auto=True, title="Trip Heatmap: Weekday × Hour", aspect="auto")
fig.update_layout(height=420)
fig.show()

## 7. Monthly Trend

In [ ]:
month_c = df.groupby("MONTH_NAME").size().reset_index(name="Trips")
month_c["MONTH_NAME"] = pd.Categorical(month_c["MONTH_NAME"], MONTH_ORDER, ordered=True)
month_c = month_c.sort_values("MONTH_NAME")
fig = px.line(month_c, x="MONTH_NAME", y="Trips", markers=True,
              title="Monthly Trip Volume (2016)")
fig.update_layout(height=380)
fig.show()

## 8. Start Locations

In [ ]:
start_c = df["START*"].value_counts().head(15).reset_index()
start_c.columns = ["Location","Count"]
fig = px.bar(start_c, x="Count", y="Location", orientation="h",
             color="Count", color_continuous_scale="Teal",
             title="Top 15 Trip Start Locations")
fig.update_layout(height=480, yaxis=dict(autorange="reversed"))
fig.show()

## 9. Miles by Purpose

In [ ]:
fig = px.box(df, x="PURPOSE*", y="MILES*", color="PURPOSE*",
             title="Distance Distribution by Trip Purpose")
fig.update_layout(height=460, xaxis_tickangle=-30, showlegend=False)
fig.show()

## 10. Avg Metrics by Purpose

In [ ]:
grp = df.groupby("PURPOSE*")[["MILES*","TRAVEL_HRS","SPEED_MPH"]].mean().reset_index()
grp.columns = ["Purpose","Avg Miles","Avg Travel (hrs)","Avg Speed (mph)"]
grp_long = grp.melt("Purpose", var_name="Metric", value_name="Value")
fig = px.bar(grp_long, x="Purpose", y="Value", color="Metric",
             barmode="group", title="Average Metrics by Trip Purpose")
fig.update_layout(height=450, xaxis_tickangle=-30)
fig.show()

## 11. Key Insights

In [ ]:
print("=== UBER RIDES KEY INSIGHTS ===")
print(f"Total trips        : {len(df):,}")
print(f"Total miles        : {df['MILES*'].sum():,.1f}")
print(f"Most common purpose: {df['PURPOSE*'].value_counts().index[0]}")
print(f"Most common category:{df['CATEGORY*'].value_counts().index[0]}")
print(f"Busiest day        : {df['WEEKDAY'].value_counts().index[0]}")
print(f"Peak hour          : {df['HOUR'].value_counts().index[0]}:00")
print(f"Top start location : {df['START*'].value_counts().index[0]}")
print(f"Short trips <5mi   : {(df['MILES*']<5).mean()*100:.1f}%")
print(f"Avg speed          : {df['SPEED_MPH'].dropna().mean():.1f} mph")